In [ ]:
from collections import defaultdict
import gensim.downloader
import gensim.models
import os
import re

def read_words(basedir="/Users/enovikov11/Desktop/data/mac/words/"):
    words = {"count": defaultdict(int), "by_file": defaultdict(set)}

    for filename in os.listdir(basedir):
        with open(basedir + filename, "r", errors="ignore", encoding="utf-8") as file:
            for line in file:
                words["count"][line.strip()] += 1
                words["by_file"][filename].add(line.strip())
    
    return words

def get_model(wordlist, model_id="glove-wiki-gigaword-50"):
    if model_id not in model_cache:
        model_cache[model_id] = gensim.downloader.load(model_id)
    
    model = model_cache[model_id]
    wordlist = [word for word in wordlist if word in model.key_to_index]

    submodel = gensim.models.KeyedVectors(vector_size=model.vector_size)
    submodel.add_vectors(wordlist, model.vectors[[model.key_to_index[word] for word in wordlist]])

    return submodel

In [ ]:
model_cache = {}
words = read_words()

In [ ]:
wordlist = [word for word, count in words["count"].items() if count > 0 and re.match(r"^[a-z]{3,}$", word, re.I)]
model = get_model(wordlist)

In [ ]:
len(model)

In [ ]:
model["hitler"] - model["germany"]

In [ ]:
def calc1(a, topn=1):
    expr = f"{a}"
    vec = model.get_vector(a)

    return expr, model.most_similar([vec], topn=topn)

def calc3(a, b, c, topn=1):
    expr = f"{a} + {b} - {c}"
    vec = model.get_vector(a) + model.get_vector(b) - model.get_vector(c)

    return expr, model.most_similar([vec], topn=topn)

In [33]:
import ast

ast.parse("1 + d").body[0].value.right

In [51]:

print(ast.dump(ast.parse("1 + (2 * -d + 3); import x"), indent=4))

Module(
    body=[
        Expr(
            value=BinOp(
                left=Constant(value=1),
                op=Add(),
                right=BinOp(
                    left=BinOp(
                        left=Constant(value=2),
                        op=Mult(),
                        right=UnaryOp(
                            op=USub(),
                            operand=Name(id='d', ctx=Load()))),
                    op=Add(),
                    right=Constant(value=3)))),
        Import(
            names=[
                alias(name='x')])],
    type_ignores=[])


In [ ]:
# queen - king = woman - man
# aunt - uncle = woman - man
# niece - nephew = woman - man
# mother - father = woman - man

# hitler + italy - germany = mussolini
# sushi + germany - japan = bratwurst

# sum (a * b)

# plur = cats - cat
# plur * puppy 
# plur * puppies

# plur * one
# plur * two
# plur * three

# 14:20 - 17:50 https://www.youtube.com/watch?v=wjZofJX0v4M

In [ ]:
best = []

In [ ]:
import random
import heapq
import json

try:
    while True:
        a, b, c = random.choice(keep), random.choice(keep), random.choice(keep)

        if a == b or b == c or c == a:
            continue

        expr, sim = calc3(a, b, c)
        result = (sim[0][1], expr + " = " + sim[0][0])

        if sim[0][0] in (a, b, c):
            continue

        if len(best) < 50:
            heapq.heappush(best, result)
        else:
            heapq.heappushpop(best, result)
except KeyboardInterrupt:
    print(json.dumps(sorted(best, key=lambda x: -x[0]), indent=2))

In [ ]:




gensim.models.KeyedVectors.load_word2vec_format("~/gensim-data/glove-wiki-gigaword-50/glove-wiki-gigaword-50.gz")

gensim.downloader.info("glove-wiki-gigaword-50")
gensim.downloader.info()